In [1]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [2]:
%cd /content/gdrive/MyDrive

/content/gdrive/MyDrive


In [3]:
import os
if not os.path.isdir("Opencv_DL"):
  os.makedirs("Opencv_DL")

In [4]:
%cd Opencv_DL

/content/gdrive/MyDrive/Opencv_DL


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from keras import datasets, layers, models
from keras.layers import Input, Dense, Flatten, Lambda
from keras.models import Model, Sequential
from keras.applications.vgg16 import VGG16
from keras.preprocessing import image
from glob import glob
from keras.preprocessing.image import ImageDataGenerator

In [6]:
Image_size = [224,224]
train_path = "/content/gdrive/MyDrive/Opencv_DL/Train"
test_path = "/content/gdrive/MyDrive/Opencv_DL/Test"

In [7]:
vgg = VGG16(input_shape=Image_size + [3], weights= "imagenet", include_top=False)
for layer in vgg.layers:
  layer.trainable = False

58889256/58889256 [==============================] - 0s 0us/step


In [8]:
folders = glob("/content/gdrive/MyDrive/Opencv_DL/Train/*")
folders

['/content/gdrive/MyDrive/Opencv_DL/Train/emilia-clarke',
 '/content/gdrive/MyDrive/Opencv_DL/Train/justin',
 '/content/gdrive/MyDrive/Opencv_DL/Train/kit-harington',
 '/content/gdrive/MyDrive/Opencv_DL/Train/nikolaj-coster-waldau',
 '/content/gdrive/MyDrive/Opencv_DL/Train/peter-dinklage']

In [9]:
x = Flatten()(vgg.output)
prediction = Dense(len(folders), activation='softmax')(x)
model = Model(inputs=vgg.input, outputs=prediction)
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [10]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics = ['accuracy'])

In [12]:
train_datagen = ImageDataGenerator(rescale = 1./255, shear_range=0.2, zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale = 1./255)

In [13]:
training_set = train_datagen.flow_from_directory("/content/gdrive/MyDrive/Opencv_DL/Train",
                                                 target_size=(224,224), batch_size=32,
                                                 class_mode="categorical")

test_set = test_datagen.flow_from_directory("/content/gdrive/MyDrive/Opencv_DL/Test",
                                                 target_size=(224,224), batch_size=32,
                                                 class_mode="categorical")

Found 32 images belonging to 5 classes.
Found 5 images belonging to 5 classes.


In [15]:
final_model = model.fit_generator(training_set, validation_data=test_set, epochs=10,
                                  steps_per_epoch=len(training_set), validation_steps=len(test_set))

<ipython-input-15-c48763644b3f>:1: UserWarning: `Model.fit_generator` is deprecated and will be removed in a future version. Please use `Model.fit`, which supports generators.
  final_model = model.fit_generator(training_set, validation_data=test_set, epochs=10,


Epoch 1/10
1/1 [==============================] - 1s 917ms/step - loss: 0.5097 - accuracy: 0.7812 - val_loss: 0.1749 - val_accuracy: 1.0000
Epoch 2/10
1/1 [==============================] - 1s 794ms/step - loss: 0.2374 - accuracy: 0.9688 - val_loss: 0.1151 - val_accuracy: 1.0000
Epoch 3/10
1/1 [==============================] - 1s 748ms/step - loss: 0.1581 - accuracy: 1.0000 - val_loss: 0.1104 - val_accuracy: 1.0000
Epoch 4/10
1/1 [==============================] - 1s 799ms/step - loss: 0.1798 - accuracy: 0.9688 - val_loss: 0.1076 - val_accuracy: 1.0000
Epoch 5/10
1/1 [==============================] - 1s 773ms/step - loss: 0.1299 - accuracy: 1.0000 - val_loss: 0.0914 - val_accuracy: 1.0000
Epoch 6/10
1/1 [==============================] - 1s 777ms/step - loss: 0.2174 - accuracy: 0.9375 - val_loss: 0.0545 - val_accuracy: 1.0000
Epoch 7/10
1/1 [==============================] - 1s 752ms/step - loss: 0.1909 - accuracy: 0.9375 - val_loss: 0.0263 - val_accuracy: 1.0000
Epoch 8/10
1/1 [====

In [16]:
from keras.models import load_model
model.save("my_own_model_cv.h5")

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [17]:
# face detection part with CNN model
from PIL import Image
import json
import cv2
from google.colab.patches import cv2_imshow

In [ ]:
def face_extractor(img):
  faces = face_casecade.detectMultiScale(img, 1.5, 5)